# Proyecto Titanic

Vamos a revisar los datos del Titanic, decidir qué limpiar y preparar las columnas para predecir si una persona sobrevivió o no.

# Integrantes

- Alfredo Avila Escobar

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# sklearn devuelve tablas de pandas, como en clase
set_config(transform_output="pandas")

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

# Primer contacto

Primero revisamos cómo vienen los datos: filas, columnas, tipos de dato, faltantes y duplicados.

- ¿Clasificación o regresión?
    **R=** Clasificación, dado que buscamos predecir si una persona sobrevive o no.

- ¿Qué numérica se comporta como categórica, y qué categórica tiene orden?
    **R=** `survived` y `pclass` son numéricas pero se comportan como categorías (0 o 1, y clase 1, 2 o 3). `class` es una categórica que tiene orden: First, Second y Third.

- ¿Qué parejas de columnas dicen lo mismo? Compruébalo.
    **R=** `survived` y `alive`; `pclass` y `class`; `embarked` y `embark_town`. La comprobación está más abajo.

- ¿Qué pasa si le das alive al modelo?
    **R=** Le estamos dando la respuesta al modelo generando Target Leakage (un tipo de Data Leakage). Esto provoca un rendimiento engañosamente perfecto (100% de precisión) durante el entrenamiento, pero hace que el modelo sea completamente inútil en producción porque esa variable no estará disponible al momento de predecir casos nuevos.

Faltantes por columna

In [ ]:
df.isna().sum() #total de nulos

Porcentaje de faltantes por columna

In [ ]:
df.isna().mean() * 100

Número de filas y columnas totales

In [ ]:
df.shape #(891, 15)

Primeras y últimas filas

In [ ]:
display(df.head())
display(df.tail())

Tipos de Dato

In [ ]:
df.dtypes

Resumen estadístico de numéricas y categóricas

In [ ]:
display(df.describe())
display(df.describe(exclude="number"))

Filas duplicadas

In [ ]:
df.duplicated().sum() #cantidad de filas repetidas

Valores únicos de cada categoría

In [ ]:
columnas_categoricas = ["sex", "embarked", "class", "who", "adult_male",
                        "deck", "embark_town", "alive", "alone", "survived", "pclass"]

for col in columnas_categoricas:
    print(f"--- Columna: {col} ---")
    print(df[col].value_counts(dropna=False), "\n")

Comprobación de columnas que dicen lo mismo

Cambiamos los códigos por sus equivalencias y contamos las coincidencias con `sum()`. Si el resultado es 891, las dos columnas coinciden en todas las filas.

In [ ]:
alive_numerico = df["alive"].map({"no": 0, "yes": 1})
(alive_numerico == df["survived"]).sum()

In [ ]:
clase_texto = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})
(clase_texto == df["class"]).sum()

In [ ]:
puerto_texto = df["embarked"].map({"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"})

# Usamos la misma etiqueta para comparar también los faltantes
(puerto_texto.fillna("Sin dato") == df["embark_town"].fillna("Sin dato")).sum()

In [ ]:
print("Coincidencias entre adult_male y who:", (df["adult_male"] == (df["who"] == "man")).sum())
print("Coincidencias entre alone y familiares:", (df["alone"] == ((df["sibsp"] + df["parch"]) == 0)).sum())

Las tres parejas coinciden en las 891 filas, así que dicen lo mismo. También vemos que `adult_male` coincide con los hombres de `who`, y que `alone` indica si no hay familiares registrados a bordo.

## Análisis exploratorio

- Si solo pudieras usar una columna para predecir, ¿cuál?
    **R=** Elegiríamos `sex`, porque en estos datos hay una diferencia grande de supervivencia entre mujeres y hombres. Todavía no entrenamos un modelo para comprobar qué columna predice mejor.

- Tres hallazgos que te sorprendan.
    **R=** Sobrevivió menos de la mitad de los pasajeros; la supervivencia fue mucho mayor en mujeres que en hombres; y en primera clase sobrevivió un porcentaje mayor que en tercera.

- ¿Algo huele a error en los datos?
    **R=** Hay filas repetidas, faltan muchas edades y datos de `deck`, y hay tarifas de cero o muy altas. Hay que revisarlos antes de decidir si son errores.

In [ ]:
(df["survived"].value_counts(normalize=True) * 100).round(2) #porcentajes de supervivientes

In [ ]:
fig, ax = plt.subplots()

sns.countplot(data=df, x="survived", ax=ax)

ax.set_title("Distribución de la supervivencia")
ax.set_xlabel("Supervivencia (0 = no, 1 = sí)")
ax.set_ylabel("Número de pasajeros")

plt.show()

Podemos observar cómo se distribuyen los datos, interpretando que la mayoría murió: 61.62% no sobrevivió y 38.38% sí. Las clases no están perfectamente equilibradas, pero la menos frecuente tiene casi 4 de cada 10 pasajeros, así que no es un desbalance fuerte. Aun así usaremos `stratify` al separar train y test.

Histogramas y boxplots de edad y tarifa

Usamos el histograma para ver cómo se reparten los valores y el boxplot para ubicar los que se alejan del resto.

In [ ]:
for col in ["age", "fare"]:
    fig, ax = plt.subplots()
    sns.histplot(data=df, x=col, bins=30, ax=ax)
    ax.set_title(f"Distribución de {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Número de pasajeros")
    plt.show()

    fig, ax = plt.subplots()
    sns.boxplot(data=df, x=col, ax=ax)
    ax.set_title(f"Boxplot de {col}")
    ax.set_xlabel(col)
    plt.show()

En `age` se concentran más pasajeros en edades adultas jóvenes. La mediana es de 28 años y también hay bebés y personas mayores; en el boxplot solo unos pocos puntos quedan arriba de 65 años. Estas gráficas no incluyen las 177 edades faltantes.

En `fare` la mayoría de las tarifas es baja y hay unas pocas mucho más altas. Por eso la media es 32.20, mientras que la mediana es 14.45. En el boxplot hay muchos puntos arriba de 65 libras.

Barras de sexo, clase, puerto y familiares

In [ ]:
for col in ["sex", "pclass", "embarked", "sibsp", "parch"]:
    fig, ax = plt.subplots()
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(f"Pasajeros por {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Número de pasajeros")
    plt.show()

- En `sex` hay más hombres: 577, frente a 314 mujeres.
- En `pclass` la mayoría viajó en tercera clase: 491 pasajeros.
- En `embarked` predomina Southampton (S), con 644 pasajeros. Los dos pasajeros sin puerto no aparecen en la gráfica.
- En `sibsp` lo más común es cero, aunque hay personas con hasta ocho hermanos o cónyuges registrados.
- En `parch` también predomina cero y el máximo es seis padres o hijos registrados.

Supervivencia por sexo, clase y si viajaba solo

In [ ]:
for col in ["sex", "pclass", "alone"]:
    porcentaje = df.groupby(col)["survived"].mean() * 100
    display(porcentaje.round(2))

    fig, ax = plt.subplots()
    ax.bar(porcentaje.index.astype(str), porcentaje.values)
    ax.set_title(f"Supervivencia por {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Supervivencia (%)")
    ax.set_ylim(0, 100)
    plt.show()

- Por sexo, sobrevivió el 74.20% de las mujeres y el 18.89% de los hombres.
- Por clase, sobrevivió el 62.96% de primera, 47.28% de segunda y 24.24% de tercera.
- Entre quienes viajaban solos sobrevivió el 30.35%, y entre quienes tenían familiares registrados, el 50.56%.

Estos porcentajes muestran diferencias entre los grupos, pero no prueban que esa sea la causa.

Supervivencia por edad y tarifa

Hacemos una copia para agrupar las edades y tarifas con `pd.cut`. Los límites de tarifa son 10, 30 y 100 libras. Son tramos que elegimos para comparar y no tienen la misma cantidad de pasajeros.

In [ ]:
df_grupos = df.copy()
df_grupos["grupo_edad"] = pd.cut(
    df_grupos["age"],
    bins=[-1, 12, 17, 59, np.inf],
    labels=["Hasta 12", "Más de 12 a 17", "Más de 17 a 59", "Más de 59"]
)
df_grupos["grupo_tarifa"] = pd.cut(
    df_grupos["fare"],
    bins=[-1, 10, 30, 100, np.inf],
    labels=["Hasta 10", "Más de 10 a 30", "Más de 30 a 100", "Más de 100"]
)

for col in ["grupo_edad", "grupo_tarifa"]:
    porcentaje = df_grupos.groupby(col)["survived"].mean() * 100
    display(porcentaje.round(2))
    display(df_grupos[col].value_counts())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(porcentaje.index.astype(str), porcentaje.values)
    ax.set_title(f"Supervivencia por {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Supervivencia (%)")
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()

En edad, el porcentaje de supervivencia es mayor en el grupo de hasta 12 años y menor en el de más de 59. Hay que tener en cuenta que faltan edades y que los grupos no tienen la misma cantidad de personas.

En tarifa, los grupos más caros tienen mayor supervivencia que el grupo de hasta 10 libras. El precio también está relacionado con la clase del pasajero.

¿Influyó el puerto?

Primero vemos la supervivencia por puerto y después lo cruzamos con clase.

In [ ]:
supervivencia_puerto = df.groupby("embarked")["survived"].mean() * 100
display(supervivencia_puerto.round(2))

fig, ax = plt.subplots()
ax.bar(supervivencia_puerto.index, supervivencia_puerto.values)
ax.set_title("Supervivencia por puerto")
ax.set_xlabel("Puerto")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)
plt.show()

En Cherbourg (C) sobrevivió el 55.36%, en Queenstown (Q) el 38.96% y en Southampton (S) el 33.70%. Revisamos también qué clases viajaban desde cada puerto.

In [ ]:
clases_puerto = pd.crosstab(df["embarked"], df["pclass"], normalize="index") * 100
display(clases_puerto.round(2))

fig, ax = plt.subplots()
sns.heatmap(clases_puerto, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=100, ax=ax)
ax.set_title("Porcentaje de cada clase por puerto")
ax.set_xlabel("Clase")
ax.set_ylabel("Puerto")
plt.show()

En Cherbourg cerca de la mitad es de primera clase, mientras que en Queenstown casi todos son de tercera. Por eso no podemos atribuir la diferencia de supervivencia solamente al puerto.

In [ ]:
supervivencia_puerto_clase = pd.pivot_table(
    df, values="survived", index="embarked", columns="pclass", aggfunc="mean"
) * 100

display(supervivencia_puerto_clase.round(2))
display(pd.crosstab(df["embarked"], df["pclass"]))

fig, ax = plt.subplots()
sns.heatmap(supervivencia_puerto_clase, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=100, ax=ax)
ax.set_title("Supervivencia por puerto y clase (%)")
ax.set_xlabel("Clase")
ax.set_ylabel("Puerto")
plt.show()

Incluso separando por clase hay diferencias entre puertos, pero algunos grupos son pequeños. Por ejemplo, en Queenstown solo hay dos pasajeros de primera y tres de segunda.

Mujeres de tercera clase frente a hombres de primera

In [ ]:
mujeres_tercera = df[(df["sex"] == "female") & (df["pclass"] == 3)]
hombres_primera = df[(df["sex"] == "male") & (df["pclass"] == 1)]

comparacion = pd.DataFrame({
    "Grupo": ["Mujeres de tercera", "Hombres de primera"],
    "Pasajeros": [len(mujeres_tercera), len(hombres_primera)],
    "Supervivencia (%)": [mujeres_tercera["survived"].mean() * 100,
                           hombres_primera["survived"].mean() * 100]
})
display(comparacion.round(2))

fig, ax = plt.subplots()
ax.bar(comparacion["Grupo"], comparacion["Supervivencia (%)"])
ax.set_title("Mujeres de tercera y hombres de primera")
ax.set_xlabel("Grupo")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)
plt.show()

Sobrevivió el 50% de las mujeres de tercera clase y el 36.89% de los hombres de primera. En esta comparación, viajar en primera no fue suficiente para tener un porcentaje mayor que el otro grupo.

## Limpieza

Copia de los datos originales

Guardamos una copia antes de cambiar algo. Lo que sigue parte de `df_original`.

In [ ]:
df_original = df.copy()

Columnas que vamos a quitar

- `alive`: da la respuesta que queremos predecir.
- `class` y `embark_town`: repiten la clase y el puerto.
- `who` y `adult_male`: resumen información de sexo y edad. Las quitamos para trabajar con esas columnas directamente.
- `deck`: faltan 688 datos de 891. Preferimos quitarla antes que rellenar la mayor parte de la columna. Otra opción sería poner una categoría de desconocido.
- `alone`: después crearemos el tamaño de familia, que también permite saber quién viajaba solo.

In [ ]:
columnas_eliminar = ["alive", "class", "embark_town", "who", "adult_male", "deck", "alone"]
df_limpio = df_original.drop(columns=columnas_eliminar).copy()
df_limpio.head()

Filas duplicadas

No hay nombre ni número de billete para saber si dos filas son la misma persona. Primero vemos cuántas se repiten y qué perfil tienen.

In [ ]:
repetidas = df_original[df_original.duplicated()]

print("Filas repetidas:", len(repetidas))
print("Hombres solos de tercera clase:", ((repetidas["sex"] == "male") & (repetidas["pclass"] == 3) & repetidas["alone"]).sum())
repetidas.head()

Hay 107 filas repetidas y más de la mitad son hombres que viajaban solos en tercera clase, un perfil muy común en el barco. Las conservamos: sin nombre ni billete no sabemos si son la misma persona o pasajeros distintos con datos iguales, y borrarlas podría quitar pasajeros reales.

Valores extremos con IQR y Z-score

Revisamos `fare`, `age`, `sibsp` y `parch` de dos formas. IQR usa la distancia entre los cuartiles y Z-score indica cuántas desviaciones estándar se aleja un valor de la media (marcamos los mayores a 3). Que un valor salga marcado no significa que esté mal.

In [ ]:
def limites_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

resultados_outliers = []

for col in ["fare", "age", "sibsp", "parch"]:
    valores = df_original[col].dropna()
    limite_inferior, limite_superior = limites_iqr(valores)
    fuera_iqr = (valores < limite_inferior) | (valores > limite_superior)

    z_score = (valores - valores.mean()) / valores.std()
    fuera_z = z_score.abs() > 3

    resultados_outliers.append({
        "Columna": col,
        "Límite inferior": limite_inferior,
        "Límite superior": limite_superior,
        "Fuera de IQR": fuera_iqr.sum(),
        "Z-score mayor a 3": fuera_z.sum()
    })

display(pd.DataFrame(resultados_outliers).round(2))

- En `fare` IQR marca 116 valores y Z-score 20. Los conservamos y después aplicamos logaritmo, para que las tarifas altas no estiren tanto la escala.
- En `age` se marcan 11 con IQR y 2 con Z-score. Las edades van de 0.42 a 80 años y son posibles, así que las conservamos.
- En `sibsp` se marcan 46 con IQR y 30 con Z-score. Una familia grande puede ser real, por eso no la quitamos solo por ser poco común.
- En `parch` se marcan 213 con IQR y 15 con Z-score. Aquí los dos cuartiles son cero, así que IQR marca cualquier valor mayor que cero. No sería una buena razón para borrarlos.

Para la pregunta de las tarifas de 0 o de más de 500, las edades menores de un año y los 8 hermanos, contamos cuántos casos hay y revisamos cómo son.

In [ ]:
print("Tarifas de cero:", (df_original["fare"] == 0).sum())
print("Tarifas mayores a 500:", (df_original["fare"] > 500).sum())
print("Valores de esas tarifas:", df_original[df_original["fare"] > 500]["fare"].unique())
print("Edades menores de un año:", (df_original["age"] < 1).sum())
print("Edades negativas:", (df_original["age"] < 0).sum())
print("Tarifas negativas:", (df_original["fare"] < 0).sum())
print("Casos con sibsp igual a 8:", (df_original["sibsp"] == 8).sum())
display(df_original[df_original["sibsp"] == 8][["pclass", "parch", "fare"]].drop_duplicates())

- ¿Eliminarías las tarifas de 0 o de más de 500, las edades menores de un año y los casos con ocho hermanos a bordo?
    **R=** No los eliminaríamos automáticamente, porque con las columnas que tenemos no podemos confirmar que estén mal.
    - Las 3 tarifas de más de 500 valen lo mismo (512.33), así que parecen billetes de un mismo grupo y no un error de captura.
    - Las 7 edades menores de un año son bebés posibles (de 0.42 a 0.92 años).
    - Los 7 casos con 8 hermanos tienen la misma clase, los mismos padres o hijos y la misma tarifa, así que parece una sola familia grande.
    - Las 15 tarifas de cero pueden ser boletos sin costo o datos que no se registraron, no sabemos cuál. Como no hay evidencia suficiente, las conservamos.

Separación de entrenamiento y prueba

Separamos los datos antes de rellenar los faltantes. Así la mediana y la moda se calculan solo con entrenamiento (`train`), sin usar los datos de prueba (`test`). Dejamos 80% para train y 20% para test, con `stratify` para mantener la misma proporción de supervivientes.

In [ ]:
X = df_limpio.drop(columns="survived")
y = df_limpio["survived"]

X_train_original, X_test_original, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Filas de train:", len(X_train_original))
print("Filas de test:", len(X_test_original))
print("Supervivencia en train (%):", round(y_train.mean() * 100, 2))
print("Supervivencia en test (%):", round(y_test.mean() * 100, 2))

El porcentaje de supervivientes queda casi igual en train (38.34%), en test (38.55%) y en los datos completos (38.38%).

Faltantes de edad y puerto

Para `age` usamos la mediana de train, porque los valores extremos la afectan menos que a la media. Antes de rellenar, creamos `edad_faltante` con `isna().astype(int)`, como en la clase de limpieza, para no perder cuáles edades faltaban. Otra opción era la mediana por grupo (por ejemplo, por sexo y clase), pero preferimos una sola mediana para mantenerlo simple.

Para `embarked` usamos la moda de train, es decir, el puerto más frecuente. No usamos la media porque los puertos son categorías.

Guardamos esos dos valores y usamos `fillna` para rellenar train y test con la misma mediana y la misma moda.

In [ ]:
X_train_limpio = X_train_original.copy()
X_test_limpio = X_test_original.copy()

X_train_limpio["edad_faltante"] = X_train_limpio["age"].isna().astype(int)
X_test_limpio["edad_faltante"] = X_test_limpio["age"].isna().astype(int)

mediana_edad = X_train_original["age"].median()
moda_puerto = X_train_original["embarked"].mode()[0]
print("Mediana de edad en train:", mediana_edad)
print("Moda de puerto en train:", moda_puerto)

In [ ]:
X_train_limpio["age"] = X_train_limpio["age"].fillna(mediana_edad)
X_test_limpio["age"] = X_test_limpio["age"].fillna(mediana_edad)

X_train_limpio["embarked"] = X_train_limpio["embarked"].fillna(moda_puerto)
X_test_limpio["embarked"] = X_test_limpio["embarked"].fillna(moda_puerto)

display(X_train_limpio.head())
display(X_train_limpio.isna().sum())
display(X_test_limpio.isna().sum())

Antes y después de limpiar

Juntamos las dos partes ya rellenadas para comparar con el original. Esta tabla es solo para revisar el resultado; train y test siguen separados para los siguientes pasos.

In [ ]:
df_comparacion = pd.concat([X_train_limpio, X_test_limpio])
df_comparacion["survived"] = y

antes_despues = pd.DataFrame({
    "Antes": [len(df_original), df_original.shape[1],
              df_original.isna().sum().sum(), df_original["survived"].mean() * 100],
    "Después": [len(df_comparacion), df_comparacion.shape[1],
                df_comparacion.isna().sum().sum(), df_comparacion["survived"].mean() * 100]
}, index=["Filas", "Columnas", "Faltantes", "Supervivencia (%)"])

display(antes_despues.round(2))

In [ ]:
for col in ["age", "fare"]:
    antes_despues_columna = pd.DataFrame({
        "Antes": df_original[col].describe(),
        "Después": df_comparacion[col].describe()
    })
    display(antes_despues_columna.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(data=df_original, x=col, bins=20, ax=axes[0])
    sns.histplot(data=df_comparacion, x=col, bins=20, ax=axes[1])
    axes[0].set_title(f"{col} antes")
    axes[1].set_title(f"{col} después")
    for ax in axes:
        ax.set_xlabel(col)
        ax.set_ylabel("Número de pasajeros")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.bar(["Antes", "Después"], [df_original["survived"].mean() * 100,
                               df_comparacion["survived"].mean() * 100])
ax.set_title("Supervivencia antes y después")
ax.set_xlabel("Momento de la revisión")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)
plt.show()

En edad aparece una barra más alta cerca de la mediana porque ahí rellenamos los faltantes, y la desviación estándar baja de 14.53 a 13.01. La distribución de tarifa queda igual y la supervivencia sigue en 38.38%, ya que no eliminamos pasajeros. Las columnas pasaron de 15 a 9 porque quitamos 7 y agregamos `edad_faltante`.

## Transformación y escalado

Nuevas variables

Creamos `family_size`, sumando los familiares y al propio pasajero, y `age_group`, separando las edades en cuatro grupos con `pd.cut`. También usaremos `edad_faltante`, que ya creamos antes de rellenar: vale 1 si la edad faltaba y 0 si estaba registrada.

Las variables nuevas se calculan con las columnas de cada pasajero, sin usar `survived`, y no aprenden nada de los datos. Por eso se calculan igual en train y en test.

In [ ]:
X_train_variables = X_train_limpio.copy()
X_test_variables = X_test_limpio.copy()

for tabla in [X_train_variables, X_test_variables]:
    tabla["family_size"] = tabla["sibsp"] + tabla["parch"] + 1
    tabla["age_group"] = pd.cut(
        tabla["age"], bins=[-1, 12, 17, 59, np.inf],
        labels=["Infancia", "Adolescencia", "Adultez", "Mayor"]
    )

X_train_variables.head()

Comparamos las variables nuevas con la supervivencia usando solo train. Los grupos de edad son hasta 12 años, más de 12 a 17, más de 17 a 59 y más de 59.

In [ ]:
df_train = X_train_variables.copy()
df_train["survived"] = y_train

for col in ["family_size", "age_group"]:
    porcentaje = df_train.groupby(col)["survived"].mean() * 100
    display(porcentaje.round(2))
    display(df_train[col].value_counts())

    fig, ax = plt.subplots()
    ax.bar(porcentaje.index.astype(str), porcentaje.values)
    ax.set_title(f"Supervivencia por {col} en train")
    ax.set_xlabel(col)
    ax.set_ylabel("Supervivencia (%)")
    ax.set_ylim(0, 100)
    plt.show()

display((df_train.groupby("edad_faltante")["survived"].mean() * 100).round(2))

En train, las familias de 2 a 4 personas tienen mayor supervivencia que quienes viajaban solos (29.95%). Los grupos de 5 o más tienen pocos pasajeros, por eso sus porcentajes cambian mucho con pocos casos.

En `age_group` hay más supervivencia en infancia (60.00%) que en adultez (36.62%) y mayores (16.67%). Hay que recordar que las edades rellenadas con la mediana quedaron en adultez.

La última tabla muestra que también hay diferencia entre quienes tenían la edad registrada (40.52%) y quienes no (29.20%).

Asimetría y logaritmo

Primero medimos la asimetría de las numéricas en train con `skew()` (0 es simétrica) y la comparamos con la que quedaría usando `log1p`.

In [ ]:
numericas_revisar = ["age", "sibsp", "parch", "family_size", "fare"]

asimetria = pd.DataFrame({
    "Original": X_train_variables[numericas_revisar].skew(),
    "Con log1p": np.log1p(X_train_variables[numericas_revisar]).skew()
})
asimetria.round(2)

In [ ]:
tarifa_log = np.log1p(X_train_variables["fare"])
print("Asimetría de fare:", round(X_train_variables["fare"].skew(), 3))
print("Asimetría con log1p:", round(tarifa_log.skew(), 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(X_train_variables["fare"], bins=30, ax=axes[0])
sns.histplot(tarifa_log, bins=30, ax=axes[1])
axes[0].set_title("Tarifa original en train")
axes[1].set_title("Tarifa con log1p en train")
axes[0].set_xlabel("Tarifa")
axes[1].set_xlabel("Logaritmo de 1 + tarifa")
for ax in axes:
    ax.set_ylabel("Número de pasajeros")
plt.tight_layout()
plt.show()

`fare` es la más asimétrica (4.65) y con `log1p` baja a 0.31. `sibsp`, `parch` y `family_size` también son asimétricas, pero son conteos donde la mayoría vale 0 o 1: con el logaritmo solo bajan a cerca de 1.5 o 1.7, y sus valores altos son familias reales. `age` ya era casi simétrica (0.45) y con el logaritmo empeora (-2.21). Por eso aplicamos el logaritmo solo a `fare`.

- ¿Qué haces con los ceros?
    **R=** Hay 15 tarifas de cero y el logaritmo de 0 no existe. Por eso usamos `np.log1p`, que calcula el logaritmo de 1 más la tarifa: un cero queda en cero y no da error.

Con el logaritmo los precios altos quedan menos separados del resto. Por ahora lo calculamos para comparar las gráficas; la transformación que usaremos al final irá en el Pipeline, como en clase.

Codificación

- Para `sex` y `embarked` usamos one-hot, porque no tienen un orden.
- Con `drop="if_binary"`, `sex` queda en una columna de 0 y 1. El puerto conserva una columna por categoría.
- `edad_faltante` también es binaria. Con el mismo codificador conserva una sola columna y no la escalamos.
- `pclass` y `age_group` tienen orden. Indicamos sus categorías en `OrdinalEncoder`: primera, segunda y tercera clase; infancia, adolescencia, adultez y mayor.

- ¿Pclass como número o one-hot?
    **R=** Elegimos ordinal para conservar el orden de las clases. One-hot sería otra opción si no queremos representar ese orden con una sola columna numérica.

- ¿Cuándo usar `drop="first"`?
    **R=** Sirve para quitar la primera columna de cada grupo one-hot, porque esa categoría se puede reconocer cuando las demás valen cero. En clase se menciona sobre todo para modelos lineales. Aquí usamos `drop="if_binary"`, que solo quita una columna cuando hay dos categorías.

Escalado y Pipeline

Usamos `StandardScaler`, que es lo habitual. `MinMaxScaler` sería mejor si necesitáramos un rango fijo entre 0 y 1, pero con valores extremos (como `sibsp` hasta 8) casi todos los datos quedarían amontonados cerca de 0. Escalar no borra los valores extremos ni corrige la asimetría; por eso `fare` pasa primero por el logaritmo.

Para `fare` encadenamos logaritmo y escalado en un Pipeline. El `ColumnTransformer` aplica ese Pipeline a la tarifa y los otros tratamientos a sus columnas, como en la clase de transformación.

In [ ]:
numericas = ["age", "sibsp", "parch", "family_size"]
asimetricas = ["fare"]
nominales = ["sex", "embarked", "edad_faltante"]
ordinales = ["pclass", "age_group"]

pipeline_asimetrico = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("escalar", StandardScaler())
])

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), numericas),
    ("asim", pipeline_asimetrico, asimetricas),
    ("nom", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False), nominales),
    ("ord", OrdinalEncoder(categories=[[1, 2, 3], ["Infancia", "Adolescencia", "Adultez", "Mayor"]]), ordinales)
])

preprocesador

`fit_transform` aprende la codificación y las escalas usando train. A test solo le aplicamos `transform`. Los faltantes ya se rellenaron con valores de train y las variables nuevas se crearon en las celdas anteriores.

In [ ]:
X_train = preprocesador.fit_transform(X_train_variables)
X_test = preprocesador.transform(X_test_variables)

display(X_train.head().round(3))

columnas_escaladas = ["num__age", "num__sibsp", "num__parch", "num__family_size", "asim__fare"]
display(pd.DataFrame({
    "Media en train": X_train[columnas_escaladas].mean(),
    "Media en test": X_test[columnas_escaladas].mean(),
    "Desviación en train": X_train[columnas_escaladas].std()
}).round(3))

Las columnas escaladas de train quedan con media cercana a cero y desviación cercana a uno. En test la media no tiene por qué ser cero, porque usamos las escalas aprendidas con train.

- ¿En qué orden van split, imputación y escalado?
    **R=** Primero separamos train y test. Después calculamos cómo rellenar, codificar y escalar usando solo train. A test le aplicamos los mismos pasos con los valores aprendidos.

- ¿Qué pasa si calculas la mediana de age con todos los datos?
    **R=** Estaríamos usando información de test para preparar train. Eso es fuga de información, aunque no usemos directamente la respuesta `survived`.

Bonus: todo dentro de un Pipeline

Hasta aquí rellenamos los faltantes a mano para ver la mediana y la moda. Ahora hacemos lo mismo con `SimpleImputer` dentro del `ColumnTransformer`, y todo dentro de un `Pipeline`. Partimos de train y test sin rellenar.

Dejamos fuera `age_group`, porque se calcula con la edad ya rellenada. `edad_faltante` y `family_size` sí se pueden crear antes, porque no dependen de rellenar.

In [ ]:
X_train_bonus = X_train_original.copy()
X_test_bonus = X_test_original.copy()

for tabla in [X_train_bonus, X_test_bonus]:
    tabla["edad_faltante"] = tabla["age"].isna().astype(int)
    tabla["family_size"] = tabla["sibsp"] + tabla["parch"] + 1

pipeline_edad = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler())
])

pipeline_puerto = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocesador_bonus = ColumnTransformer([
    ("edad", pipeline_edad, ["age"]),
    ("num", StandardScaler(), ["sibsp", "parch", "family_size"]),
    ("asim", pipeline_asimetrico, ["fare"]),
    ("puerto", pipeline_puerto, ["embarked"]),
    ("nom", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False), ["sex", "edad_faltante"]),
    ("ord", OrdinalEncoder(categories=[[1, 2, 3]]), ["pclass"])
])

pipeline_completo = Pipeline([("preprocesar", preprocesador_bonus)])

X_train_bonus = pipeline_completo.fit_transform(X_train_bonus)
X_test_bonus = pipeline_completo.transform(X_test_bonus)

print("X_train_bonus:", X_train_bonus.shape)
print("X_test_bonus:", X_test_bonus.shape)

Comparamos con lo que hicimos a mano: si el resultado es `True`, los números son iguales.

In [ ]:
imputador_edad = pipeline_completo.named_steps["preprocesar"].named_transformers_["edad"].named_steps["imputar"]
print("Mediana que aprendió el imputador:", imputador_edad.statistics_[0], "| la que calculamos a mano:", mediana_edad)

print("Edad igual en train:", np.allclose(X_train_bonus["edad__age"], X_train["num__age"]))
print("Edad igual en test:", np.allclose(X_test_bonus["edad__age"], X_test["num__age"]))
print("Tarifa igual en train:", np.allclose(X_train_bonus["asim__fare"], X_train["asim__fare"]))
print("Puerto S igual en train:", np.allclose(X_train_bonus["puerto__embarked_S"], X_train["nom__embarked_S"]))
print("Faltantes en el bonus:", X_train_bonus.isna().sum().sum(), "en train y", X_test_bonus.isna().sum().sum(), "en test")

Salen los mismos números que con el método manual. La ventaja del Pipeline es que `fit_transform` en train aprende todo (mediana, moda, escalas y categorías) y `transform` en test solo lo aplica, sin pasos a mano. Si más adelante agregamos un modelo, iría como último paso del Pipeline. En lo que sigue usamos `X_train` y `X_test`, que sí incluyen `age_group`.

Verificación final

Revisamos faltantes, tipos de dato, columnas y tamaños de los cuatro conjuntos.

In [ ]:
print("Faltantes en train:", X_train.isna().sum().sum())
print("Faltantes en test:", X_test.isna().sum().sum())
print("Todo numérico (float64):", (X_train.dtypes == "float64").all() and (X_test.dtypes == "float64").all())
print("Mismas columnas en train y test:", list(X_train.columns) == list(X_test.columns))
print("survived en las columnas:", "survived" in X_train.columns or "survived" in X_test.columns)
print("alive en las columnas:", any("alive" in col for col in X_train.columns))
display(X_train.dtypes)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Columnas finales:", list(X_train.columns))

## Bitácora de decisiones

En afectados anotamos cuántos datos se revisan o cambian. No eliminamos filas.

| # | Columna | Problema | Opciones | Decisión | Por qué | Afectados |
|---|---|---|---|---|---|---|
| 1 | Copia original | Necesitamos comparar | Cambiar directamente o copiar | Guardar `df_original` | Poder revisar cómo venían los datos | 891 filas |
| 2 | alive | Da la respuesta | Conservar o quitar | Quitar | Evitar fuga de información | Columna completa |
| 3 | class | Repite pclass | Elegir una o dejar ambas | Quitar class | La clase ya está en pclass | Columna completa |
| 4 | embark_town | Repite embarked | Elegir una o dejar ambas | Quitar embark_town | El puerto ya está en embarked | Columna completa |
| 5 | who y adult_male | Resumen sexo y edad | Conservar o quitar | Quitar | Trabajar con sexo y edad directamente | Dos columnas |
| 6 | alone | Indica si viaja solo | Conservar o quitar | Quitar | Lo veremos también en family_size | Columna completa |
| 7 | deck | Faltan muchos datos | Rellenar, poner desconocido o quitar | Quitar | Faltaba más del 77% | 688 faltantes |
| 8 | Duplicados | 107 filas repetidas | Borrar o conservar | Conservar | No sabemos si son pasajeros distintos; más de la mitad son hombres solos de tercera | 107 filas; ninguna borrada |
| 9 | Split | Necesitamos datos de prueba | Separación simple o estratificada | 80/20 con stratify y semilla 42 | Mantener proporciones parecidas de supervivencia | 712 train y 179 test |
| 10 | age | 177 faltantes | Media, mediana, mediana por grupo o borrar | Mediana de train | Los extremos la afectan menos que a la media | 177 faltantes |
| 11 | embarked | Dos faltantes | Moda, desconocido o borrar | Moda de train | Usar el puerto más frecuente | Dos faltantes |
| 12 | edad_faltante | Se pierde cuáles edades faltaban | Crear indicador o no | isna().astype(int) antes de rellenar | Distinguir las edades rellenadas; al codificar queda una sola columna | 177 valores con indicador 1 |
| 13 | fare | Valores muy altos y tarifas de cero | Borrar, recortar o conservar | Conservar | Un valor raro no confirma un error | IQR marca 116 y Z-score 20; ninguna borrada |
| 14 | age | Edades alejadas del resto | Borrar, recortar o conservar | Conservar | Bebés y mayores pueden ser pasajeros reales | IQR marca 11 y Z-score 2 |
| 15 | sibsp | Familias grandes | Borrar o conservar | Conservar | Son conteos posibles | IQR marca 46 y Z-score 30 |
| 16 | parch | IQR marca cualquier valor mayor que cero | Borrar o conservar | Conservar | Tener familiares no es un error | IQR marca 213 y Z-score 15 |
| 17 | family_size | Queremos ver el tamaño familiar | Conteos separados o suma | sibsp + parch + 1 | Incluir al pasajero y sus familiares | 891 valores nuevos |
| 18 | age_group | Queremos comparar grupos de edad | Edad continua o tramos | Cuatro tramos con pd.cut | Comparar infancia, adolescencia, adultez y mayores | 891 valores nuevos |
| 19 | fare | Asimetría de 4.65 en train y 15 ceros | Dejarla, log o log1p | log1p | Baja la asimetría y acepta los ceros | Columna completa |
| 20 | sex y embarked | Son categorías sin orden | Asignar números o one-hot | One-hot con drop="if_binary" | Una sola columna para sexo y tres para el puerto | 891 filas codificadas |
| 21 | pclass | Tiene orden | One-hot u ordinal | Ordinal con orden 1, 2, 3 | Representar las clases en una sola columna | 891 filas codificadas |
| 22 | age_group | Tiene orden | One-hot u ordinal | Ordinal | Mantener el orden de los grupos de edad | 891 filas codificadas |
| 23 | Numéricas | Tienen escalas distintas | StandardScaler o MinMaxScaler | StandardScaler | Es lo habitual y min-max se afecta por los valores extremos | Ajuste con 712 filas y aplicación a 891 |
| 24 | Pipeline | La tarifa necesita dos pasos | Pasos separados o encadenados | Logaritmo y después escalado | Aplicar ambos pasos en el mismo orden | 712 para aprender y 179 solo para transformar |
| 25 | Bonus | Rellenar a mano no queda dentro del Pipeline | A mano o SimpleImputer | SimpleImputer dentro del Pipeline | Mismo resultado y todo se aprende solo con train | Edad y puerto: 177 y 2 faltantes |

## Conclusiones

- ¿Cuáles fueron los tres factores que más influyeron?
    **R=** Sexo, clase y tarifa. Sobrevivió el 74.20% de las mujeres y el 18.89% de los hombres; el 62.96% de primera clase y el 24.24% de tercera; y el 73.58% de quienes pagaron más de 100 libras frente al 19.94% de quienes pagaron hasta 10. No entrenamos un modelo, así que hablamos de lo que observamos en los datos. Además, clase y tarifa están relacionadas, por lo que parte de su efecto se mezcla.

- ¿Cuál fue la decisión más difícil y qué pasaría al elegir otra opción?
    **R=** Conservar los duplicados. Si los borramos, quitamos filas que podrían ser pasajeros distintos. Para decidir mejor necesitaríamos nombres, identificadores o números de billete. Al conservarlos también puede haber datos iguales en train y test, aunque sus índices sean distintos.

- ¿Qué dato faltó?
    **R=** Nos faltaron datos para identificar a cada pasajero y entender los billetes de cero o muy caros. También sería mejor tener las edades reales: rellenarlas con una mediana no las recupera. Otra opción sería usar medianas por sexo y clase, calculadas solo con train.

- ¿Cómo quedaron los datos y qué columnas verá el modelo?
    **R=** Quedaron 712 filas de train y 179 de test, con 12 columnas en cada uno. No hay faltantes y todas las entradas son numéricas. `survived` queda separado en `y_train` y `y_test`; `alive` se eliminó.

Las columnas finales son `num__age`, `num__sibsp`, `num__parch`, `num__family_size`, `asim__fare`, `nom__sex_male`, `nom__embarked_C`, `nom__embarked_Q`, `nom__embarked_S`, `nom__edad_faltante_1`, `ord__pclass` y `ord__age_group`.

Los nombres tienen prefijos como en clase: `num__` para numéricas escaladas, `asim__` para la tarifa con logaritmo y escalado, `nom__` para one-hot y `ord__` para ordinales. El sexo queda representado en una sola columna binaria. El notebook termina con los datos preparados; no entrenamos un modelo.